In [1]:
import pandas as pd

# Load AmazonHelp conversation data
conversations = pd.read_csv(
    "../data/processed/amazon_conversations.csv"
)

print("Rows:", len(conversations))
print("Columns:", conversations.columns.tolist())

display(conversations.head(10))

Rows: 98875
Columns: ['conversation_id', 'message_count', 'customer_messages', 'amazon_messages', 'is_multi_turn']


,conversation_id,message_count,customer_messages,amazon_messages,is_multi_turn
0,965004,159,99,60,True
1,179525,98,60,38,True
2,440269,80,37,43,True
3,1279825,75,40,35,True
4,219588,71,36,35,True
5,430037,69,47,22,True
6,341543,69,41,28,True
7,127995,68,36,32,True
8,197648,63,31,32,True
9,197994,61,30,31,True


In [2]:
# Load the original Amazon support messages
amazon_messages = pd.read_csv(
    "../data/processed/amazon_support.csv"
)

print("Rows:", len(amazon_messages))
print("Columns:", amazon_messages.columns.tolist())

display(
    amazon_messages[
        [
            "tweet_id",
            "inbound",
            "text",
            "in_response_to_tweet_id",
            "response_tweet_id"
        ]
    ].head(10)
)

Rows: 270343
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,inbound,text,in_response_to_tweet_id,response_tweet_id
0,648405,False,"@274086 Sorry, I'm not quite sure what you're ...",648407.0,648406
1,648406,True,@AmazonHelp here this on my gift card activity...,648405.0,648408
2,369266,False,@203483 We generally send an e-mail asking for...,369267.0,369261
3,369261,True,@AmazonHelp Here's why it matters: I need veri...,369266.0,369259
4,369259,False,@203483 The book has to be purchased on our we...,369261.0,369260
5,369260,True,@AmazonHelp That's exactly what she did. Bough...,369259.0,369262
6,369262,False,"@203483 Sorry, we would be unable to change th...",369260.0,369263
7,369263,True,@AmazonHelp Can she delete her review &amp; re...,369262.0,369264
8,130745,False,@145514 Our customers' security is our utmost ...,130747.0,130746
9,130746,True,@115850 @AmazonHelp ur __email__ seems not wor...,130745.0,130748


In [3]:
# Create customer → AmazonHelp response pairs

# Create a lookup from tweet_id to message
message_lookup = amazon_messages.set_index("tweet_id")["text"].to_dict()

pairs = []

for _, row in amazon_messages.iterrows():

    # We only want AmazonHelp replies
    if row["inbound"] != False:
        continue

    parent_id = row["in_response_to_tweet_id"]

    # Skip messages without a parent
    if pd.isna(parent_id):
        continue

    # Find the customer message that this AmazonHelp message replies to
    customer_message = message_lookup.get(int(parent_id))

    if customer_message is None:
        continue

    pairs.append({
        "customer_message": customer_message,
        "amazon_response": row["text"],
        "response_tweet_id": row["tweet_id"],
        "customer_tweet_id": int(parent_id)
    })

retrieval_df = pd.DataFrame(pairs)

print("Customer → AmazonHelp pairs:", len(retrieval_df))

display(retrieval_df.head(10))

Customer → AmazonHelp pairs: 70965


,customer_message,amazon_response,response_tweet_id,customer_tweet_id
0,@AmazonHelp Here's why it matters: I need veri...,@203483 The book has to be purchased on our we...,369259,369261
1,@AmazonHelp That's exactly what she did. Bough...,"@203483 Sorry, we would be unable to change th...",369262,369260
2,@amazonhelp No i have not recived any email li...,@376508 As requested earlier please do fill ou...,1087359,1087360
3,@AmazonHelp I have share my details,"@376508 Hey, we've received your details. We'r...",1087353,1087355
4,@amazonhelp what the hell you guys are doing,"@376508 Hi there, the concerned team is lookin...",1087356,1087354
5,@amazonhelp seller already received package be...,@15547 Hi! you may file a claim against the se...,313818,313819
6,@amazonhelp I already A2Z claims 2 times on se...,@15547 Hi there! kindly refer to the email sen...,313814,313816
7,@AmazonHelp Well not that I know near me. \nWh...,@340939 Our Account Specialists will be able t...,931346,931348
8,@AmazonHelp what is the use of Amazon? If I ne...,"@561970 Hey, i understand your concern. Amazon...",1883131,1883132
9,@AmazonHelp I'm using a Windows 10 PC,@111354 Have you tried uninstalling &amp; rein...,2902685,2902686


In [4]:
# Clean customer → AmazonHelp response pairs

before = len(retrieval_df)

# Remove missing values
retrieval_df = retrieval_df.dropna(
    subset=["customer_message", "amazon_response"]
).copy()

# Normalize whitespace
retrieval_df["customer_message"] = (
    retrieval_df["customer_message"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

retrieval_df["amazon_response"] = (
    retrieval_df["amazon_response"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove very short messages/responses
retrieval_df = retrieval_df[
    (retrieval_df["customer_message"].str.len() >= 15) &
    (retrieval_df["amazon_response"].str.len() >= 15)
].copy()

# Remove exact duplicate customer-response pairs
retrieval_df = retrieval_df.drop_duplicates(
    subset=["customer_message", "amazon_response"]
).reset_index(drop=True)

after = len(retrieval_df)

print("Pairs before cleaning:", before)
print("Pairs after cleaning:", after)
print("Removed:", before - after)

display(retrieval_df.head(10))

Pairs before cleaning: 70965
Pairs after cleaning: 70734
Removed: 231


,customer_message,amazon_response,response_tweet_id,customer_tweet_id
0,@AmazonHelp Here's why it matters: I need veri...,@203483 The book has to be purchased on our we...,369259,369261
1,@AmazonHelp That's exactly what she did. Bough...,"@203483 Sorry, we would be unable to change th...",369262,369260
2,@amazonhelp No i have not recived any email li...,@376508 As requested earlier please do fill ou...,1087359,1087360
3,@AmazonHelp I have share my details,"@376508 Hey, we've received your details. We'r...",1087353,1087355
4,@amazonhelp what the hell you guys are doing,"@376508 Hi there, the concerned team is lookin...",1087356,1087354
5,@amazonhelp seller already received package be...,@15547 Hi! you may file a claim against the se...,313818,313819
6,@amazonhelp I already A2Z claims 2 times on se...,@15547 Hi there! kindly refer to the email sen...,313814,313816
7,@AmazonHelp Well not that I know near me. Why ...,@340939 Our Account Specialists will be able t...,931346,931348
8,@AmazonHelp what is the use of Amazon? If I ne...,"@561970 Hey, i understand your concern. Amazon...",1883131,1883132
9,@AmazonHelp I'm using a Windows 10 PC,@111354 Have you tried uninstalling &amp; rein...,2902685,2902686


In [5]:
retrieval_df.to_csv(
    "../data/processed/amazon_retrieval_pairs.csv",
    index=False
)

print("Saved: data/processed/amazon_retrieval_pairs.csv")

Saved: data/processed/amazon_retrieval_pairs.csv


In [6]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
retrieval_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [7]:
import numpy as np

customer_texts = retrieval_df["customer_message"].tolist()

retrieval_embeddings = retrieval_model.encode(
    customer_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", retrieval_embeddings.shape)

Batches:   0%|          | 0/1106 [00:00<?, ?it/s]

Embedding shape: (70734, 384)


In [8]:
import os
import numpy as np

os.makedirs("../models", exist_ok=True)

np.save(
    "../models/retrieval_embeddings.npy",
    retrieval_embeddings
)

print("Saved:", "../models/retrieval_embeddings.npy")
print("Shape:", retrieval_embeddings.shape)

Saved: ../models/retrieval_embeddings.npy
Shape: (70734, 384)


In [10]:
from sklearn.neighbors import NearestNeighbors

retrieval_index = NearestNeighbors(
    n_neighbors=5,
    metric="cosine",
    algorithm="brute"
)

retrieval_index.fit(retrieval_embeddings)

print("Retrieval index built.")

Retrieval index built.


In [11]:
query = "Where is my package? It was supposed to arrive yesterday."

# Convert query into an embedding
query_embedding = retrieval_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Find 5 most similar historical customer messages
distances, indices = retrieval_index.kneighbors(query_embedding)

for rank, (distance, idx) in enumerate(
    zip(distances[0], indices[0]), start=1
):
    row = retrieval_df.iloc[idx]

    print(f"\n--- Result {rank} ---")
    print(f"Similarity: {1 - distance:.4f}")
    print(f"Customer: {row['customer_message']}")
    print(f"AmazonHelp: {row['amazon_response']}")


--- Result 1 ---
Similarity: 0.8280
Customer: @AmazonHelp It says it was delivered yesterday at 10pm. I was at home and no one delivered anything?!!! Where is my package
AmazonHelp: @324180 Hi, have you checked with close neighbours &amp; safe places around property: https://t.co/BuAkojNX8g ^TS

--- Result 2 ---
Similarity: 0.7376
Customer: @AmazonHelp It says delivered yesterday handed to resident. I wasn't in all day yesterday so that's bollocks. Where's my parcel?
AmazonHelp: @164971 As we're unable to view your account via Twitter, pls contact here: https://t.co/JzP7hlA23B so we can look further into this.^KM

--- Result 3 ---
Similarity: 0.7237
Customer: @AmazonHelp I ordered a package on Friday to be delivered today and it’s not here. Now it’s not projected to be here til Wednesday or Thursday
AmazonHelp: @797329 I'm sorry about the wait! Orders need a bit more time to fulfill and ship due to Black Friday and Cyber Monday. Thanks for your continued patience with us! ^SH

--- Res